In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F 
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-dataframe-preparation").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 14:24:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


## What is a Spark DataFrame?

A **DataFrame** is a distributed table with named, typed columns -- conceptually like a Pandas DataFrame or SQL table, but distributed across a cluster.

DataFrames are built on top of RDDs but provide two major advantages:
1. **Catalyst optimizer**: Spark rewrites and optimizes your query plan before executing it
2. **Tungsten engine**: columnar in-memory storage with code generation for CPU efficiency

`.explain()` shows Spark's execution plan -- the parsed, analyzed, optimized, and physical plan -- useful for understanding and debugging performance.

# Example 1: DataFrame Basics


## DataFrame vs RDD

| Feature | RDD | DataFrame |
|---------|-----|-----------|
| API | Functional (`map`, `filter`) | SQL / column expressions |
| Type safety | Any Python object | Schema-based (StructType) |
| Query optimization | None | Catalyst + Tungsten |
| Best for | Unstructured data, custom objects | Structured / tabular data |

`df.where(df['age'] > 1)` and `df.rdd.filter(lambda r: r['age'] > 1)` produce the same result  
but through very different execution paths -- the DataFrame version goes through the optimizer.

In [2]:
df = spark.createDataFrame([('Alice', 1), ('Bob', 2)], ['name', 'age'])
df

DataFrame[name: string, age: bigint]

In [3]:
df.show()

+-----+---+
| name|age|
+-----+---+
|Alice|  1|
|  Bob|  2|
+-----+---+



In [4]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [5]:
df.rdd.collect()

[Row(name='Alice', age=1), Row(name='Bob', age=2)]

In [6]:
df.rdd.filter(lambda r: r['age'] > 1).collect()

[Row(name='Bob', age=2)]

In [7]:
df.where(df['age'] > 1).show()

+----+---+
|name|age|
+----+---+
| Bob|  2|
+----+---+



In [8]:
df2 = df.where(df['age'] > 1)

In [9]:
df2.explain()

== Physical Plan ==
*(1) Filter (isnotnull(age#1L) AND (age#1L > 1))
+- *(1) Scan ExistingRDD[name#0,age#1L]




In [10]:
df2.explain(True)

== Parsed Logical Plan ==
'Filter '`>`(age#1L, 1)
+- LogicalRDD [name#0, age#1L], false

== Analyzed Logical Plan ==
name: string, age: bigint
Filter (age#1L > cast(1 as bigint))
+- LogicalRDD [name#0, age#1L], false

== Optimized Logical Plan ==
Filter (isnotnull(age#1L) AND (age#1L > 1))
+- LogicalRDD [name#0, age#1L], false

== Physical Plan ==
*(1) Filter (isnotnull(age#1L) AND (age#1L > 1))
+- *(1) Scan ExistingRDD[name#0,age#1L]



In [11]:
df2.explain("extended")

== Parsed Logical Plan ==
'Filter '`>`(age#1L, 1)
+- LogicalRDD [name#0, age#1L], false

== Analyzed Logical Plan ==
name: string, age: bigint
Filter (age#1L > cast(1 as bigint))
+- LogicalRDD [name#0, age#1L], false

== Optimized Logical Plan ==
Filter (isnotnull(age#1L) AND (age#1L > 1))
+- LogicalRDD [name#0, age#1L], false

== Physical Plan ==
*(1) Filter (isnotnull(age#1L) AND (age#1L > 1))
+- *(1) Scan ExistingRDD[name#0,age#1L]



In [12]:
df2.explain("codegen")

Found 1 WholeStageCodegen subtrees.
== Subtree 1 / 1 (maxMethodCodeSize:237; maxConstantPoolSize:124(0.19% used); numInnerClasses:0) ==
*(1) Filter (isnotnull(age#1L) AND (age#1L > 1))
+- *(1) Scan ExistingRDD[name#0,age#1L]

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Iterator[] inputs;
/* 009 */   private scala.collection.Iterator rdd_input_0;
/* 010 */   private org.apache.spark.sql.catalyst.expressions.codegen.UnsafeRowWriter[] rdd_mutableStateArray_0 = new org.apache.spark.sql.catalyst.expressions.codegen.UnsafeRowWriter[2];
/* 011 */
/* 012 */   public GeneratedIteratorForCodegenStage1(Object[] references) {
/* 013 */     this.references = r

# Example 2: Tweet Analysis

## Reading Multi-Line CSV

Standard CSV parsers break when a field contains a newline character. Spark handles this with  
`.option("multiLine", "true")` -- but this forces a single-partition read (slower for large files).

The Biden tweet dataset uses quoted fields with embedded newlines, so `multiLine` is required here.

In [13]:
file_name = os.path.join(os.getcwd(), "data/joe_biden_tweets_2020.csv")
data = spark.read.option("header", "true").option("multiLine", "true").option("inferSchema", "true").csv(file_name)
data

DataFrame[timestamp: string, url: string, tweet: string, replies: string, retweets: string, quotes: string, likes: string]

In [14]:
data.printSchema()

root
 |-- timestamp: string (nullable = true)
 |-- url: string (nullable = true)
 |-- tweet: string (nullable = true)
 |-- replies: string (nullable = true)
 |-- retweets: string (nullable = true)
 |-- quotes: string (nullable = true)
 |-- likes: string (nullable = true)



In [15]:
data.count()

2987

In [16]:
data.show() # show the first 20 rows

+----------------+--------------------+--------------------+-------+--------+------+-----+
|       timestamp|                 url|               tweet|replies|retweets|quotes|likes|
+----------------+--------------------+--------------------+-------+--------+------+-----+
|2020-01-01 01:15|https://twitter.c...|Our final fundrai...|    411|     269|    32|  948|
|2020-01-01 18:35|https://twitter.c...|Every single huma...|   1136|    2423|   182|11574|
|2020-01-02 00:01|https://twitter.c...|With just over on...|    332|     368|    29| 1457|
|2020-01-02 01:05|https://twitter.c...|This election is ...|   5199|   10192|  1153|44886|
|2020-01-02 02:07|https://twitter.c...|Every day that Do...|   1070|    2005|   128| 9581|
|2020-01-02 16:10|https://twitter.c...|It was a privileg...|    439|    2284|    90|17156|
|2020-01-02 17:55|https://twitter.c...|Like Vicky said, ...|    780|    1297|   105| 6657|
|2020-01-02 19:01|https://twitter.c...|I'm excited to sh...|   1499|    2351|   326|14346|

In [17]:
data.show(5) # show the first 5 rows

+----------------+--------------------+--------------------+-------+--------+------+-----+
|       timestamp|                 url|               tweet|replies|retweets|quotes|likes|
+----------------+--------------------+--------------------+-------+--------+------+-----+
|2020-01-01 01:15|https://twitter.c...|Our final fundrai...|    411|     269|    32|  948|
|2020-01-01 18:35|https://twitter.c...|Every single huma...|   1136|    2423|   182|11574|
|2020-01-02 00:01|https://twitter.c...|With just over on...|    332|     368|    29| 1457|
|2020-01-02 01:05|https://twitter.c...|This election is ...|   5199|   10192|  1153|44886|
|2020-01-02 02:07|https://twitter.c...|Every day that Do...|   1070|    2005|   128| 9581|
+----------------+--------------------+--------------------+-------+--------+------+-----+
only showing top 5 rows


In [18]:
first5 = data.take(5)

In [19]:
type(first5)

list

In [20]:
first5

[Row(timestamp='2020-01-01 01:15', url='https://twitter.com/JoeBiden/status/1212180387260010496', tweet='Our final fundraising deadline of 2019 is just hours away and we need your help. Every donation — big or small — goes a long way to helping us make Donald Trump a one-term president.\n\nChip in before midnight to help us reach our goal: https://t.co/CznFJFHT2T', replies='411', retweets='269', quotes='32', likes='948'),
 Row(timestamp='2020-01-01 18:35', url='https://twitter.com/JoeBiden/status/1212442112219844609', tweet='Every single human being deserves to be treated with dignity. Everyone. The poor and the powerless, the marginalized and vulnerable, the “least of these.” That has been the animating principle of my life and my faith. https://t.co/BwmOVQjxVk', replies='1136', retweets='2423', quotes='182', likes='11574'),
 Row(timestamp='2020-01-02 00:01', url='https://twitter.com/JoeBiden/status/1212524152608833536', tweet="With just over one month until the Iowa Caucus, we need a

In [21]:
first5[0]

Row(timestamp='2020-01-01 01:15', url='https://twitter.com/JoeBiden/status/1212180387260010496', tweet='Our final fundraising deadline of 2019 is just hours away and we need your help. Every donation — big or small — goes a long way to helping us make Donald Trump a one-term president.\n\nChip in before midnight to help us reach our goal: https://t.co/CznFJFHT2T', replies='411', retweets='269', quotes='32', likes='948')

In [22]:
first5[0]['tweet']

'Our final fundraising deadline of 2019 is just hours away and we need your help. Every donation — big or small — goes a long way to helping us make Donald Trump a one-term president.\n\nChip in before midnight to help us reach our goal: https://t.co/CznFJFHT2T'

In [23]:
# select tweets in 2020-11
novTweets = data.where(data.timestamp.startswith("2020-11"))
novTweets.count()

19

In [24]:
# how many tweets contain @realDonaldTrump (in any casing)
trumpTweets = data.filter(F.lower( F.col("tweet")).contains("@realdonaldtrump"))
trumpTweets.count()
                          

15

In [25]:
trumpTweets.show()

+----------------+--------------------+--------------------+-------+--------+------+------+
|       timestamp|                 url|               tweet|replies|retweets|quotes| likes|
+----------------+--------------------+--------------------+-------+--------+------+------+
|2020-01-05 19:40|https://twitter.c...|Hey @realDonaldTr...|   1308|    3235|   178| 14803|
|2020-01-15 23:05|https://twitter.c...|This seems like a...|    343|    1595|    37|  8544|
|2020-02-05 20:20|https://twitter.c...|Fred, it must not...|   1038|    8017|   338| 44879|
|2020-02-15 00:32|https://twitter.c...|Wanted to make su...|   1521|   13961|   627| 40497|
|2020-03-04 04:07|https://twitter.c...|You lost tonight,...|  12111|   14722|  2244| 94241|
|2020-07-20 19:17|https://twitter.c...|President Trump a...|   4401|   26672|  2227| 76038|
|2020-08-25 02:20|https://twitter.c...|.@realDonaldTrump...|   4453|   15283|   780| 66386|
|2020-09-03 00:30|https://twitter.c...|I’ve released 21 ...|  19493|   54944|  4

In [26]:
# ok, that displayed all the line, extract the tweet text
trumpTweetsText = trumpTweets.select(F.col("tweet"))
trumpTweetsText.show()

+--------------------+
|               tweet|
+--------------------+
|Hey @realDonaldTr...|
|This seems like a...|
|Fred, it must not...|
|Wanted to make su...|
|You lost tonight,...|
|President Trump a...|
|.@realDonaldTrump...|
|I’ve released 21 ...|
|More than 1,000 p...|
|.@realDonaldTrump...|
|.@realDonaldTrump...|
|"Let me be very c...|
|Show us your tax ...|
|.@realDonaldTrump...|
|Yesterday, @realD...|
+--------------------+



In [27]:
records1 = trumpTweetsText.collect()


In [28]:
r = records1[0]
r

Row(tweet='Hey @realDonaldTrump, do you think the American people want another war in the Middle East? Des Moines certainly doesn’t. https://t.co/uFgQuqg1oL')

In [29]:
r['tweet']

'Hey @realDonaldTrump, do you think the American people want another war in the Middle East? Des Moines certainly doesn’t. https://t.co/uFgQuqg1oL'

In [30]:
text = list(map(lambda r: r['tweet'], records1))
text

['Hey @realDonaldTrump, do you think the American people want another war in the Middle East? Des Moines certainly doesn’t. https://t.co/uFgQuqg1oL',
 'This seems like a great reason to rejoin the Paris Agreement, @realDonaldTrump. https://t.co/HHI4KUokcF',
 'Fred, it must not have been easy to sit there and listen to @realdonaldtrump lie. I’m proud of you for speaking truth to the abuse of power. https://t.co/xrnSAKC8QZ',
 'Wanted to make sure you saw this, @realDonaldTrump: “Trump’s First 3 Years Created 1.5 Million Fewer Jobs Than Obama’s Last 3.” https://t.co/KWoHe1d4ku',
 'You lost tonight, @realDonaldTrump. Democrats around the country are fired up. We are decent, brave, and resilient people. We are better than you. Come November, we are going to beat you. https://t.co/gjZcrcCDlX',
 'President Trump asked to see the COVID death chart. @RealDonaldTrump, here you go. https://t.co/45tYD2NXti',
 '.@realDonaldTrump it didn’t have to be this bad. https://t.co/aMhxaTXnpm',
 'I’ve releas

In [31]:
# how many tweets in each month?
df = data.withColumn("month", F.substring(F.col("timestamp"), 0, 7))
dfCount = df.groupBy(F.col("month")).count().orderBy(F.col("count").desc())
dfCount

DataFrame[month: string, count: bigint]

In [32]:
dfCount.show()

+-------+-----+
|  month|count|
+-------+-----+
|2020-10|  504|
|2020-08|  337|
|2020-03|  324|
|2020-09|  303|
|2020-02|  277|
|2020-04|  266|
|2020-05|  250|
|2020-07|  228|
|2020-06|  219|
|2020-01|  201|
|2020-11|   19|
|#DemCon|    4|
|We can'|    2|
|https:/|    2|
|    Now|    2|
|America|    2|
|But the|    2|
|We're p|    2|
|- Lead |    1|
|On Frid|    1|
+-------+-----+
only showing top 20 rows


In [33]:
r = dfCount.first()
topMonth, topCount= r

In [34]:
topMonth

'2020-10'

In [35]:
topCount

504

# Example 3: DataFrame Schema

## Schema Inference vs Manual Schema

| | Inferred (`inferSchema=true`) | Manual (`StructType`) |
|--|------|-----|
| Convenience | High -- Spark auto-detects types | Low -- must specify all fields |
| Correctness | Medium -- may mistype (e.g., dates as strings) | High -- you control every type |
| Performance | Slightly slower (needs a scan pass) | Faster -- no preliminary scan |

**Always define a manual schema in production** for correctness and performance.  
The `StructType` / `StructField` API mirrors SQL `CREATE TABLE` syntax.

In [37]:
# reading a data frame
file_name = os.path.join(os.getcwd(), "data/cars.json")
firstDF = spark.read.format("json").option("inferSchema", "true").load(file_name)

In [38]:
# displaying the data frame
firstDF.show()

+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|Acceleration|Cylinders|Displacement|Horsepower|Miles_per_Gallon|                Name|Origin|Weight_in_lbs|      Year|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|        12.0|        8|       307.0|       130|            18.0|chevrolet chevell...|   USA|         3504|1970-01-01|
|        11.5|        8|       350.0|       165|            15.0|   buick skylark 320|   USA|         3693|1970-01-01|
|        11.0|        8|       318.0|       150|            18.0|  plymouth satellite|   USA|         3436|1970-01-01|
|        12.0|        8|       304.0|       150|            16.0|       amc rebel sst|   USA|         3433|1970-01-01|
|        10.5|        8|       302.0|       140|            17.0|         ford torino|   USA|         3449|1970-01-01|
|        10.0|        8|       429.0|       198|

In [39]:
# printing the schema
firstDF.printSchema()

root
 |-- Acceleration: double (nullable = true)
 |-- Cylinders: long (nullable = true)
 |-- Displacement: double (nullable = true)
 |-- Horsepower: long (nullable = true)
 |-- Miles_per_Gallon: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Weight_in_lbs: long (nullable = true)
 |-- Year: string (nullable = true)



In [42]:
r = firstDF.first()
r

Row(Acceleration=12.0, Cylinders=8, Displacement=307.0, Horsepower=130, Miles_per_Gallon=18.0, Name='chevrolet chevelle malibu', Origin='USA', Weight_in_lbs=3504, Year='1970-01-01')

In [43]:
r.Year

'1970-01-01'

In [44]:
r['Year']

'1970-01-01'

In [45]:
inferredSchema = firstDF.schema
inferredSchema

StructType([StructField('Acceleration', DoubleType(), True), StructField('Cylinders', LongType(), True), StructField('Displacement', DoubleType(), True), StructField('Horsepower', LongType(), True), StructField('Miles_per_Gallon', DoubleType(), True), StructField('Name', StringType(), True), StructField('Origin', StringType(), True), StructField('Weight_in_lbs', LongType(), True), StructField('Year', StringType(), True)])

In [46]:
manualSchema = T.StructType([
    T.StructField("Name", T.StringType(), True),
    T.StructField("Miles_per_Gallon", T.DoubleType(), True),
    T.StructField("Cylinders", T.LongType(), True),
    T.StructField("Displacement", T.DoubleType(), True),
    T.StructField("Horsepower", T.LongType(), True),
    T.StructField("Weight_in_lbs", T.LongType(), True),
    T.StructField("Acceleration", T.DoubleType(), True),
    T.StructField("Year", T.DateType(), True),   # <----
    T.StructField("Origin", T.StringType(), True)
])
manualSchema

StructType([StructField('Name', StringType(), True), StructField('Miles_per_Gallon', DoubleType(), True), StructField('Cylinders', LongType(), True), StructField('Displacement', DoubleType(), True), StructField('Horsepower', LongType(), True), StructField('Weight_in_lbs', LongType(), True), StructField('Acceleration', DoubleType(), True), StructField('Year', DateType(), True), StructField('Origin', StringType(), True)])

In [47]:
secondDF = spark.read.format("json").schema(manualSchema).load("data/cars.json")

In [48]:
secondDF.show()

+--------------------+----------------+---------+------------+----------+-------------+------------+----------+------+
|                Name|Miles_per_Gallon|Cylinders|Displacement|Horsepower|Weight_in_lbs|Acceleration|      Year|Origin|
+--------------------+----------------+---------+------------+----------+-------------+------------+----------+------+
|chevrolet chevell...|            18.0|        8|       307.0|       130|         3504|        12.0|1970-01-01|   USA|
|   buick skylark 320|            15.0|        8|       350.0|       165|         3693|        11.5|1970-01-01|   USA|
|  plymouth satellite|            18.0|        8|       318.0|       150|         3436|        11.0|1970-01-01|   USA|
|       amc rebel sst|            16.0|        8|       304.0|       150|         3433|        12.0|1970-01-01|   USA|
|         ford torino|            17.0|        8|       302.0|       140|         3449|        10.5|1970-01-01|   USA|
|    ford galaxie 500|            15.0|        8

In [50]:
r = secondDF.first()
r

Row(Name='chevrolet chevelle malibu', Miles_per_Gallon=18.0, Cylinders=8, Displacement=307.0, Horsepower=130, Weight_in_lbs=3504, Acceleration=12.0, Year=datetime.date(1970, 1, 1), Origin='USA')

In [51]:
r.Year

datetime.date(1970, 1, 1)

In [52]:
# because Year is date we can perform calendar operations
df2 = secondDF.withColumn("age", F.months_between(F.current_date(), F.col("Year")))
df2.show()

+--------------------+----------------+---------+------------+----------+-------------+------------+----------+------+------------+
|                Name|Miles_per_Gallon|Cylinders|Displacement|Horsepower|Weight_in_lbs|Acceleration|      Year|Origin|         age|
+--------------------+----------------+---------+------------+----------+-------------+------------+----------+------+------------+
|chevrolet chevell...|            18.0|        8|       307.0|       130|         3504|        12.0|1970-01-01|   USA|677.32258065|
|   buick skylark 320|            15.0|        8|       350.0|       165|         3693|        11.5|1970-01-01|   USA|677.32258065|
|  plymouth satellite|            18.0|        8|       318.0|       150|         3436|        11.0|1970-01-01|   USA|677.32258065|
|       amc rebel sst|            16.0|        8|       304.0|       150|         3433|        12.0|1970-01-01|   USA|677.32258065|
|         ford torino|            17.0|        8|       302.0|       140|   

In [53]:
spark.stop()